# Train the transformer memory (H18) on Colab

Runs the experiments listed in **Settings** one after another (A100: ~20 min each).

**Before the first run (once):**
1. Upload `tlfp-data.zip` to Google Drive as `MyDrive/tlfp/tlfp-data.zip`.
2. In Colab: **Runtime → Change runtime type → T4 GPU**.
3. Add your Weights & Biases key: click the 🔑 **Secrets** icon in the left bar, add a secret named `WANDB_API_KEY`, and turn on notebook access. (Your key is at wandb.ai/authorize.)

**Each session:** Runtime → Run all. Checkpoints are saved to `MyDrive/tlfp/checkpoints` at every evaluation. If the session disconnects, just Run all again: training resumes where it stopped.

Keep this browser tab open; free Colab disconnects idle sessions.

## Settings

In [ ]:
# Runs, in order. Each resumes by itself if the session drops; finished runs are skipped.
# BASE is the first transformer run's setup; each run adds or overrides flags (later flags win).
BASE = ("--data data/stage3-100k data/stage3-100k-b data/stage3-backtrack data/stage3-backtrack-b "
        "--k 4 --copy-prior --split-entity --fates --unet --lr 3e-4 --dino-weight 20 "
        "--memory 128 --stride 8 --mem-patch 9 --revealed-weight 10 "
        "--init checkpoints/best-so-far.pt --batch 32 --amp")
RUNS = [
    # (name, extra flags, epochs)
    ("s3-tf-world",    "--world", 2),                     # H18 option B: world positions
    ("s3-tf-noreveal", "--revealed-weight 1", 2),         # H19: without the revealed-cell weight
    ("s3-tf-more",     "--init /content/drive/MyDrive/tlfp/checkpoints/s3-tf-mem128-last.pt --lr 2e-4", 3),  # longer training
]
SPEED_CHECK_ONLY = False  # True: just measure speed, don't train

## Setup

In [ ]:
!nvidia-smi -L
from google.colab import drive, userdata
drive.mount("/content/drive")
import os
DRIVE = "/content/drive/MyDrive/tlfp"
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)

In [ ]:
%cd /content
!test -d repo || git clone -q https://github.com/jschomay/thunder-lizard-next-frame-prediction.git repo
%cd /content/repo
!git pull -q && git log --oneline -1
!pip -q install wandb

In [ ]:
# data: copy the zip from Drive and unpack it (local disk is much faster than reading from Drive)
!test -d data/stage3-100k || (cp {DRIVE}/tlfp-data.zip /content/ && unzip -q /content/tlfp-data.zip -d /content/repo)
!ls data && ls -la checkpoints

In [ ]:
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

## Speed check (about 2 minutes)
Prints seconds per step and an estimate of minutes per epoch.

In [ ]:
!python -m tlfp.train {BASE} --epochs 1 --max-steps 120 --skip-start-eval --no-wandb --name speed --out-dir /content/speed 2>&1 | grep -v -i warn | tail -3

## Train (resumes automatically after a disconnect)

In [ ]:
import subprocess, torch
for name, extra, epochs in RUNS:
    if SPEED_CHECK_ONLY:
        break
    last = f"{DRIVE}/checkpoints/{name}-last.pt"
    if os.path.exists(last) and torch.load(last, map_location="cpu", weights_only=False).get("finished"):
        print(f"== {name}: already finished, skipping"); continue
    resume = f"--resume {last}" if os.path.exists(last) else ""
    print(f"== {name}: " + ("resuming" if resume else "fresh start"), flush=True)
    cmd = f"python -m tlfp.train {BASE} {extra} --epochs {epochs} --name {name} --out-dir {DRIVE}/checkpoints {resume}"
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "warn" not in line.lower():
            print(line, end="")
    p.wait()